**CAP182-SS3 - Part B - STADIOEquities account-activation model**  
**Student:** Bernette Lesejane - PGDip Data Science, STADIO (SOIT)

# 01 - Dataset loading

## Purpose
This notebook loads and reviews the 12 client tables before building the activation model.

- Check which tables are relevant, partly used, or not used
- Load only the useful tables
- Review the target definition and class balance
- Audit data quality issues before preprocessing

## Business problem
The main question is:

> Will this account make its first deposit within 90 days of registration?

Target: `activated` in `table01_registration_onboarding.csv`  
`activated = 1` when a first deposit occurs within 90 days; otherwise `0`.

In [28]:
# Install the notebook's required packages in the active Python environment.
%pip install -q xgboost imbalanced-learn seaborn

# Databricks restarts Python so newly installed packages are available.
# In other notebook environments, restart the kernel if imports fail after installation.

try:
    dbutils.library.restartPython()
except NameError:
    print("Not in Databricks — restart the kernel manually after installs complete.")

Note: you may need to restart the kernel to use updated packages.
Not in Databricks — restart the kernel manually after installs complete.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
# Setup, imports and shared settimgs

import os
import re
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xgboost as xgb
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from scipy import stats
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, auc, brier_score_loss,
                            classification_report, confusion_matrix, f1_score,
                            precision_recall_curve, precision_score, recall_score,
                            roc_auc_score, roc_curve)
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Load and audit all 12 tables
# Check relevance and remove leakage
# Review target balance and data quality

# Configure project paths and shared settings
_CURRENT_DIR = Path.cwd().resolve()
_PROJECT_ROOT = next(
    (parent for parent in (_CURRENT_DIR, *_CURRENT_DIR.parents)
     if (parent / "data" / "table01_registration_onboarding.csv").is_file()),
    None,
)
BASE = Path(os.environ.get("SS3_BASE", _PROJECT_ROOT or _CURRENT_DIR)).expanduser().resolve()


warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

if _PROJECT_ROOT is not None:
    DATA_DIR = _PROJECT_ROOT / "data"
elif os.environ.get("SS3_DATA_DIR"):
    DATA_DIR = Path(os.environ["SS3_DATA_DIR"]).expanduser().resolve()
else:
    raise FileNotFoundError(
        f"Could not locate data/table01_registration_onboarding.csv from {_CURRENT_DIR}. "
        "Open the notebook from within the repository or set SS3_DATA_DIR to the data folder."
    )
ARTIFACTS = BASE / "artifacts"    # cleaned data, split assignments, fitted preprocessor, models
RESULTS = BASE / "results"        # metrics, tables and figures
FIGS = RESULTS / "figures"
for _p in (ARTIFACTS, RESULTS, FIGS):
    _p.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TARGET = "activated"
ID = "account_id"


def read_table(name):
    """Read a client table exactly as supplied.

    keep_default_na=False stops pandas turning the literal category "None"
    (e.g. investor_experience_level = None, 1,250 rows) into a missing value.
    Only genuinely empty cells become NaN.
    """
    return pd.read_csv(DATA_DIR / f"{name}.csv", keep_default_na=False, na_values=[""])

print("Setup complete. Data folder:", DATA_DIR.resolve())

# Load and inventory all client tables
TABLE_NAMES = [p.stem for p in sorted(DATA_DIR.glob("table*.csv"))]
if not TABLE_NAMES:
    raise FileNotFoundError(
        f"No table*.csv files found in {DATA_DIR.resolve()}. "
        "Set SS3_DATA_DIR to the directory containing the client CSV files."
    )
raw = {name: read_table(name) for name in TABLE_NAMES}

inventory = pd.DataFrame([
    {
        "table": name,
        "rows": len(df),
        "columns": df.shape[1],
        "distinct_accounts": df[ID].nunique() if ID in df else np.nan,
        "rows_per_account (max)": df[ID].value_counts().max() if ID in df else np.nan,
        "exact_duplicate_rows": int(df.duplicated().sum()),
    }
    for name, df in raw.items()
])
inventory.to_csv(RESULTS / "01_table_inventory.csv", index=False)
print(f"Tables found: {len(raw)}")
inventory

# Record table relevance and leakage decisions
RELEVANCE = pd.DataFrame([
    ("table01_registration_onboarding", "USE", "Target + onboarding funnel; drop first_deposit_date, days_to_first_deposit (leakage)."),
    ("table02_initial_session_behaviour", "USE", "First-session and early engagement (pre-activation behaviour)."),
    ("table03_demographics_identity", "USE", "Stated goal, experience and region. gender excluded as a protected attribute."),
    ("table04_communication_support", "USE", "Onboarding / promotional e-mail engagement and support-contact counts."),
    ("table05_pre_deposit_interaction", "USE (partial)", "Drop pre_deposit_window_days, total_pre_deposit_minutes (censored by deposit); account_balance_before_first_action is constant 0."),
    ("table06_inapp_messaging_response", "USE", "Push and in-app engagement: a lever the client can pull."),
    ("table07_support_tickets", "NOT USED", "No timestamp (cannot prove pre-deposit); counts duplicate Table 04 exactly."),
    ("table08_marketing_campaigns", "USE (partial)", "Aggregated to account level; drop conversion_within_30days, campaign_roi_percent (outcome-derived)."),
    ("table09_compliance_kyc", "USE", "KYC/AML status; snapshot at extract date, tested in sensitivity analysis."),
    ("table10_feedback_satisfaction", "NOT USED", "barrier_to_activation exists only for non-activators (leakage); ~31% response, non-random."),
    ("table11_device_technical", "USE", "Sign-up device, OS, browser, connection, VPN / ad-blocker flags."),
    ("table12_post_deposit_activity", "NOT USED", "Exists only for activated accounts; post-deposit behaviour (leakage)."),
], columns=["table", "verdict", "reason"])
RELEVANCE.to_csv(RESULTS / "01_table_relevance.csv", index=False)

# Select the tables for analysis
used_tables = RELEVANCE.loc[RELEVANCE.verdict.str.startswith("USE"), "table"].tolist()
excluded_tables = RELEVANCE.loc[RELEVANCE.verdict.str.startswith("NOT"), "table"].tolist()
print("Tables used     :", len(used_tables))
print("Tables not used :", excluded_tables)
RELEVANCE

used = {name: raw[name] for name in used_tables}

population = set(used["table01_registration_onboarding"][ID])
for name, df in used.items():
    ids = set(df[ID])
    print(f"{name:<36} rows={len(df):>5}  accounts={len(ids):>5}  "
          f"in-target-population={len(ids & population):>5}")

t1_raw = used["table01_registration_onboarding"]
t1 = t1_raw.drop_duplicates()
assert t1[ID].is_unique, "duplicate account_id remains after exact-duplicate removal"

# Measure unique accounts and target balance
class_counts = t1[TARGET].value_counts().sort_index()
summary = pd.DataFrame({
    "class": ["Not activated (0)", "Activated (1)"],
    "accounts": [class_counts.get(0, 0), class_counts.get(1, 0)],
})
summary["share"] = summary["accounts"] / summary["accounts"].sum()

print(f"Raw rows in table01        : {len(t1_raw):,}")
print(f"Exact duplicates removed   : {len(t1_raw) - len(t1):,}")
print(f"Modelling instances        : {len(t1):,} unique accounts")
print(f"Majority : minority ratio  : {class_counts.max() / class_counts.min():.2f} : 1")
print()
print(summary.to_string(index=False, formatters={"share": "{:.1%}".format}))

summary.to_csv(RESULTS / "01_class_balance.csv", index=False)

# Audit missing values
missing_rows = []
for name, df in used.items():
    for col in df.columns:
        n_missing = int(df[col].isna().sum())
        if n_missing:
            missing_rows.append({"table": name, "column": col, "missing": n_missing,
                                 "missing_share": round(n_missing / len(df), 3)})
missing_report = pd.DataFrame(missing_rows)
print("Columns with missing values (true blanks):")
print(missing_report.to_string(index=False))

# Find inconsistent category labels
def spelling_variants(series):
    s = series.dropna().astype(str)
    raw_distinct = s.nunique()
    collapsed = s.str.replace(r"\s+", " ", regex=True).str.strip().str.lower().nunique()
    return raw_distinct, collapsed

variant_rows = []
for name, df in used.items():
    for col in df.select_dtypes(include=["object", "string"]).columns:
        if col in (ID,) or col.endswith("_date") or col.endswith("_timestamp"):
            continue
        raw_n, clean_n = spelling_variants(df[col])
        if raw_n > clean_n:
            variant_rows.append({"table": name, "column": col,
                                 "distinct_raw": raw_n, "distinct_after_normalising": clean_n,
                                 "spurious_variants": raw_n - clean_n})
variant_report = pd.DataFrame(variant_rows).sort_values("spurious_variants", ascending=False)
print("\nCategory columns with spelling / casing / spacing variants:")
print(variant_report.to_string(index=False))

# Check registration date formats
patterns = {
    "ISO  yyyy-MM-dd HH:mm:ss": r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$",
    "slash yyyy/MM/dd":          r"^\d{4}/\d{2}/\d{2}$",
    "slash dd/MM/yyyy":          r"^\d{2}/\d{2}/\d{4}$",
    "slash MM/dd/yyyy HH:mm":    r"^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$",
}
reg = t1_raw["registration_date"].astype(str)
date_rows = [{"format": k, "rows": int(reg.str.match(p).sum())} for k, p in patterns.items()]
date_rows.append({"format": "unrecognised", "rows": int(len(reg) - sum(r["rows"] for r in date_rows))})
date_report = pd.DataFrame(date_rows)
print("\nRegistration date formats (table 01):")
print(date_report.to_string(index=False))

missing_report.to_csv(RESULTS / "01_missing_values.csv", index=False)
variant_report.to_csv(RESULTS / "01_category_variants.csv", index=False)
date_report.to_csv(RESULTS / "01_date_formats.csv", index=False)

# Save the dataset audit summary
dataset_summary = pd.DataFrame([
    ("Relevant tables loaded", len(used_tables)),
    ("Tables not used (relevance / leakage)", len(excluded_tables)),
    ("Raw rows, table 01", len(t1_raw)),
    ("Exact duplicate rows removed", len(t1_raw) - len(t1)),
    ("Modelling instances (unique accounts)", len(t1)),
    ("Class 0 · not activated within 90 days", int(class_counts.get(0, 0))),
    ("Class 1 · activated within 90 days", int(class_counts.get(1, 0))),
    ("Majority : minority ratio", round(class_counts.max() / class_counts.min(), 3)),
], columns=["item", "value"])
dataset_summary.to_csv(RESULTS / "01_dataset_summary.csv", index=False)
print("Saved results/01_dataset_summary.csv")
dataset_summary

Setup complete. Data folder: C:\Users\lesejabm\Documents\GitHub\CAP182_Bernette\data
Tables found: 12
Tables used     : 9
Tables not used : ['table07_support_tickets', 'table10_feedback_satisfaction', 'table12_post_deposit_activity']
table01_registration_onboarding      rows= 4920  accounts= 4906  in-target-population= 4906
table02_initial_session_behaviour    rows= 4906  accounts= 4906  in-target-population= 4906
table03_demographics_identity        rows= 4906  accounts= 4906  in-target-population= 4906
table04_communication_support        rows= 4906  accounts= 4906  in-target-population= 4906
table05_pre_deposit_interaction      rows= 4906  accounts= 4906  in-target-population= 4906
table06_inapp_messaging_response     rows= 4906  accounts= 4906  in-target-population= 4906
table08_marketing_campaigns          rows= 8013  accounts= 4906  in-target-population= 4906
table09_compliance_kyc               rows= 4906  accounts= 4906  in-target-population= 4906
table11_device_technical      

,item,value
0,Relevant tables loaded,9.000
1,Tables not used (relevance / leakage),3.000
2,"Raw rows, table 01",4920.000
3,Exact duplicate rows removed,14.000
4,Modelling instances (unique accounts),4906.000
5,Class 0 · not activated within 90 days,2813.000
6,Class 1 · activated within 90 days,2093.000
7,Majority : minority ratio,1.344


# 02 - Data preparation

## Purpose
Create one clean row per account and prepare the modelling dataset with a proper train/validation/test split.

## Workflow
1. Standardise category labels
2. Coerce and validate numeric fields
3. Remove leakage and non-predictive features
4. Aggregate marketing data to account level
5. Join tables and fill structural blanks
6. Build engagement features
7. Split into 70/15/15, stratified on the target
8. Fit preprocessing from the training set only

> Key fix from SS2: all transformations are learned only on the training data, and the target comes from the client `activated` field.

In [30]:
# Clean categories, fix types, and remove leakage
# Build a single account level modelling dataset

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

# Reuse the paths configured in the setup cell
ARTIFACTS = BASE / "artifacts"
RESULTS = BASE / "results"
FIGS = RESULTS / "figures"
for _p in (ARTIFACTS, RESULTS, FIGS):
    _p.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TARGET = "activated"
ID = "account_id"

def read_table(name):
    """Read a client table; only truly empty cells become NaN (keeps the category 'None')."""
    return pd.read_csv(DATA_DIR / f"{name}.csv", keep_default_na=False, na_values=[""])

# Load the selected client tables
USED_TABLES = [
    "table01_registration_onboarding",
    "table02_initial_session_behaviour",
    "table03_demographics_identity",
    "table04_communication_support",
    "table05_pre_deposit_interaction",
    "table06_inapp_messaging_response",
    "table08_marketing_campaigns",
    "table09_compliance_kyc",
    "table11_device_technical",
]
raw = {name: read_table(name) for name in USED_TABLES}
for name, df in raw.items():
    print(f"{name:<36} {df.shape}")

# Standardise category labels
CATEGORICAL_COLS = [
    "acquisition_channel", "onboarding_completion_status", "onboarding_dropout_step",
    "registration_day_of_week", "device_type", "product_exploration_depth", "age_group",
    "gender", "province", "metro_status", "employment_status", "investor_experience_level",
    "stated_investment_goal", "support_ticket_type", "kyc_status", "pep_list_check",
    "sanctions_list_check", "signup_device_type", "signup_os_name", "signup_browser_name",
    "connection_type", "timezone_offset", "campaign_channel",
]

def clean_text(v):
    if isinstance(v, str):
        v = re.sub(r"\s+", " ", v).strip()
        return v if v != "" else np.nan
    return v

def canonicalise(series):
    """Map case / spacing variants onto the most frequent spelling of each value."""
    cleaned = series.map(clean_text)
    key = cleaned.map(lambda v: v.lower() if isinstance(v, str) else np.nan)
    pairs = pd.DataFrame({"k": key, "s": cleaned}).dropna(subset=["k"])
    best = pairs.groupby("k")["s"].agg(lambda x: x.value_counts().index[0])
    return key.map(best)

canon_log = []
for name, df in raw.items():
    for col in df.columns:
        if col in CATEGORICAL_COLS:
            before = df[col].nunique()
            df[col] = canonicalise(df[col])
            canon_log.append({"table": name, "column": col,
                              "distinct_before": before, "distinct_after": df[col].nunique()})
canon_log = pd.DataFrame(canon_log)
canon_log = canon_log[canon_log.distinct_before != canon_log.distinct_after]
canon_log.to_csv(RESULTS / "02_canonicalisation_log.csv", index=False)
print("Columns whose spelling was normalised:")
canon_log.to_string(index=False)

# Remove exact duplicate accounts
t1 = raw["table01_registration_onboarding"].drop_duplicates()
assert t1[ID].is_unique
print(f"table01 after exact-duplicate removal: {len(t1):,} accounts")

# Coerce numeric values and check valid ranges
NUMERIC_COLS = {
    "table01_registration_onboarding": ["registration_hour", "time_to_onboarding_completion",
                                        "num_onboarding_sessions", "activated"],
    "table02_initial_session_behaviour": ["first_session_duration_seconds", "screens_viewed_in_first_session",
                                          "education_content_viewed", "pricing_page_viewed", "faq_viewed",
                                          "promoted_feature_viewed", "first_7day_session_count",
                                          "first_30day_session_count"],
    "table04_communication_support": ["onboarding_emails_sent", "onboarding_email_open_rate",
                                      "promotional_emails_sent", "promotional_email_open_rate",
                                      "support_tickets_opened", "support_resolution_time_hours",
                                      "in_app_help_viewed", "feedback_submitted"],
    "table05_pre_deposit_interaction": ["search_page_visits", "watchlist_created", "watchlist_items",
                                        "detailed_stock_views", "price_alert_set", "comparison_tool_used",
                                        "trading_simulation_engaged", "account_balance_before_first_action",
                                        "deposit_page_views"],
    "table06_inapp_messaging_response": ["app_notification_opt_in", "push_notifications_received",
                                         "push_notification_open_rate", "in_app_promo_banner_impressions",
                                         "in_app_promo_banner_clicks", "premium_subscription_offer_shown",
                                         "premium_subscription_offer_clicked", "feature_request_submitted"],
    "table09_compliance_kyc": ["kyc_days_to_verification", "kyc_rejection_count", "aml_risk_score",
                               "compliance_flags_count", "document_expired"],
    "table11_device_technical": ["device_switch_count", "vpn_detected", "ad_blocker_detected"],
}
raw["table01_registration_onboarding"] = t1
coerced_report = []
for name, cols in NUMERIC_COLS.items():
    df = raw[name]
    for c in cols:
        before_missing = df[c].isna().sum()
        df[c] = pd.to_numeric(df[c], errors="coerce")
        coerced_report.append({"table": name, "column": c,
                               "new_missing_from_coercion": int(df[c].isna().sum() - before_missing)})
coerced_report = pd.DataFrame(coerced_report)
print("New missing values created by numeric coercion (should be 0):",
      int(coerced_report.new_missing_from_coercion.sum()))

checks = {
    "registration_hour in 0-23": t1.registration_hour.between(0, 23).all(),
    "num_onboarding_sessions >= 1": (t1.num_onboarding_sessions >= 1).all(),
    "push_notification_open_rate in [0,1]": raw["table06_inapp_messaging_response"].push_notification_open_rate.dropna().between(0, 1).all(),
    "onboarding_email_open_rate in [0,1]": raw["table04_communication_support"].onboarding_email_open_rate.dropna().between(0, 1).all(),
    "promotional_email_open_rate in [0,1]": raw["table04_communication_support"].promotional_email_open_rate.dropna().between(0, 1).all(),
    "promo clicks <= impressions": (raw["table06_inapp_messaging_response"].in_app_promo_banner_clicks <= raw["table06_inapp_messaging_response"].in_app_promo_banner_impressions).all(),
    "support_resolution_time_hours >= 0": (raw["table04_communication_support"].support_resolution_time_hours.dropna() >= 0).all(),
}
for label, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {label}")
    assert ok, f"range check failed: {label}"

# Document and remove leakage fields
DROPPED = pd.DataFrame([
    ("table01", "first_deposit_date", "Outcome leakage (exists only because the account deposited)"),
    ("table01", "days_to_first_deposit", "Outcome leakage"),
    ("table01", "registration_date", "Raw timestamp in 4 formats (ambiguous); day-of-week and hour used instead"),
    ("table01", "onboarding_completion_date", "Raw timestamp; duration kept as time_to_onboarding_completion"),
    ("table02", "first_session_date", "Raw timestamp"),
    ("table03", "gender", "Protected attribute, not used as a predictor (fairness); not required for the business question"),
    ("table05", "pre_deposit_window_days", "Measured to the deposit for activated accounts (leakage)"),
    ("table05", "total_pre_deposit_minutes", "Duration measured to the deposit (leakage)"),
    ("table05", "account_balance_before_first_action", "Constant 0 for every account: no information"),
    ("table08", "conversion_within_30days", "Outcome-derived conversion (leakage)"),
    ("table08", "campaign_roi_percent", "Derived from the conversion outcome (leakage)"),
    ("table08", "user_exposed_date", "Raw timestamp; replaced by aggregation"),
    ("table08", "marketing_campaign_id", "Identifier; replaced by aggregation"),
    ("table08", "campaign_message_theme", "Multi-valued per account; not aggregated in this version (limitation)"),
    ("table08", "is_first_touch_attribution", "Multi-valued per account; not aggregated in this version (limitation)"),
], columns=["table", "column", "reason"])
DROPPED.to_csv(RESULTS / "02_columns_dropped.csv", index=False)
DROPPED

# Aggregate campaign activity by account
mkt = raw["table08_marketing_campaigns"]
mkt_agg = (
    mkt.groupby(ID)
       .agg(campaign_exposures=("marketing_campaign_id", "size"),
            campaign_clicks=("user_clicked", "sum"),
            mean_campaign_spend_zar=("campaign_spend_per_user_zar", "mean"),
            n_campaign_channels=("campaign_channel", "nunique"))
       .reset_index()
)
mkt_agg["campaign_click_rate"] = mkt_agg["campaign_clicks"] / mkt_agg["campaign_exposures"]
print(f"Campaign rows: {len(mkt):,}  ->  accounts exposed: {len(mkt_agg):,}")
mkt_agg.head()

# Apply documented fills for structural blanks
FILLS = {
    "onboarding_dropout_step": "Completed (no dropout)",
    "support_ticket_type": "No ticket",
    "connection_type": "Unknown",
    "gender": "Not stated",
    "investor_experience_level": "Not stated",
    "stated_investment_goal": "Not stated",
    "product_exploration_depth": "Not stated",
}
for name, df in raw.items():
    for col, val in FILLS.items():
        if col in df.columns:
            df[col] = df[col].fillna(val)

t1 = raw["table01_registration_onboarding"]
t1["time_to_onboarding_completion"] = t1["time_to_onboarding_completion"].fillna(0)

t4 = raw["table04_communication_support"]
t4["support_resolution_time_hours"] = t4["support_resolution_time_hours"].fillna(0)

t9 = raw["table09_compliance_kyc"]
t9["kyc_verification_recorded"] = t9["kyc_days_to_verification"].notna().astype(int)
t9["kyc_days_to_verification"] = t9["kyc_days_to_verification"].fillna(0)

# Join tables and create account-level features
base_cols = ["account_id", "registration_day_of_week", "registration_hour", "acquisition_channel",
             "onboarding_completion_status", "time_to_onboarding_completion",
             "num_onboarding_sessions", "onboarding_dropout_step", TARGET]
modelling = t1[base_cols].copy()

join_specs = {
    "table02_initial_session_behaviour": ["first_session_duration_seconds", "screens_viewed_in_first_session",
        "education_content_viewed", "product_exploration_depth", "pricing_page_viewed", "faq_viewed",
        "promoted_feature_viewed", "first_7day_session_count", "first_30day_session_count",
        "app_version_at_signup", "device_type"],
    "table03_demographics_identity": ["age_group", "province", "metro_status", "employment_status",
        "investor_experience_level", "stated_investment_goal"],
    "table04_communication_support": ["onboarding_emails_sent", "onboarding_email_open_rate",
        "promotional_emails_sent", "promotional_email_open_rate", "support_tickets_opened",
        "support_ticket_type", "support_resolution_time_hours", "in_app_help_viewed", "feedback_submitted"],
    "table05_pre_deposit_interaction": ["search_page_visits", "watchlist_created", "watchlist_items",
        "detailed_stock_views", "price_alert_set", "comparison_tool_used", "trading_simulation_engaged",
        "deposit_page_views"],
    "table06_inapp_messaging_response": ["app_notification_opt_in", "push_notifications_received",
        "push_notification_open_rate", "in_app_promo_banner_impressions", "in_app_promo_banner_clicks",
        "premium_subscription_offer_shown", "premium_subscription_offer_clicked", "feature_request_submitted"],
    "table09_compliance_kyc": ["kyc_status", "kyc_days_to_verification", "kyc_verification_recorded",
        "kyc_rejection_count", "aml_risk_score", "pep_list_check", "sanctions_list_check",
        "compliance_flags_count", "document_expired"],
    "table11_device_technical": ["signup_device_type", "signup_os_name", "signup_browser_name",
        "connection_type", "device_switch_count", "vpn_detected", "ad_blocker_detected", "timezone_offset"],
}
for table, cols in join_specs.items():
    src = raw[table][[ID] + cols]
    before = len(modelling)
    modelling = modelling.merge(src, on=ID, how="left", validate="one_to_one")
    assert len(modelling) == before, f"row count changed after joining {table}"
    print(f"joined {table:<36} -> {modelling.shape}")

modelling = modelling.merge(mkt_agg, on=ID, how="left", validate="one_to_one")
for c in ["campaign_exposures", "campaign_clicks", "mean_campaign_spend_zar",
          "n_campaign_channels", "campaign_click_rate"]:
    modelling[c] = modelling[c].fillna(0)   # no exposure = 0, a real state

modelling["app_major_version"] = modelling.pop("app_version_at_signup").str.split(".").str[0].astype(float)

print("\nModelling table:", modelling.shape, "| accounts unique:", modelling[ID].is_unique)
print("Columns still containing NaN:", modelling.columns[modelling.isna().any()].tolist())

modelling["Pre_Deposit_Engagement_Count"] = (
    modelling["search_page_visits"] + modelling["watchlist_items"] + modelling["detailed_stock_views"]
    + modelling["price_alert_set"] + modelling["comparison_tool_used"]
    + modelling["trading_simulation_engaged"] + modelling["deposit_page_views"]
)
modelling["Expected_Push_Opens"] = (
    modelling["push_notifications_received"] * modelling["push_notification_open_rate"]
)
impr = modelling["in_app_promo_banner_impressions"]
modelling["Promo_Click_Rate"] = np.where(impr > 0, modelling["in_app_promo_banner_clicks"] / impr.where(impr > 0, 1), 0.0)
f30 = modelling["first_30day_session_count"]
modelling["Early_Session_Share"] = modelling["first_7day_session_count"] / (f30 + 1)

assert TARGET not in modelling.drop(columns=[TARGET]).columns
print("Engineered features added:", ["Pre_Deposit_Engagement_Count", "Expected_Push_Opens",
                                     "Promo_Click_Rate", "Early_Session_Share"])
modelling[["Pre_Deposit_Engagement_Count", "Expected_Push_Opens", "Promo_Click_Rate", "Early_Session_Share"]].describe().round(3)

# Check for remaining leakage columns
FORBIDDEN = ["first_deposit_date", "days_to_first_deposit", "conversion_within_30days",
             "campaign_roi_percent", "barrier_to_activation", "pre_deposit_window_days",
             "total_pre_deposit_minutes", "gender", "first_deposit_amount_zar",
             "post_deposit_session_count_90d", "total_trades_90days", "account_balance_90days"]
leaked = [c for c in FORBIDDEN if c in modelling.columns]
assert not leaked, f"leakage columns present: {leaked}"
print("Leakage check passed: no forbidden column is in the modelling table.")

modelling = modelling.set_index(ID)
# Split accounts into train, validation, and test sets
y_all = modelling[TARGET].astype(int)
X_all = modelling.drop(columns=[TARGET])

X_trval, X_test, y_trval, y_test = train_test_split(
    X_all, y_all, test_size=0.15, stratify=y_all, random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(
    X_trval, y_trval, test_size=0.15 / 0.85, stratify=y_trval, random_state=RANDOM_STATE)

split_table = pd.DataFrame([
    {"subset": "Training", "accounts": len(X_train), "activated": int(y_train.sum()),
     "activation_rate": y_train.mean()},
    {"subset": "Validation", "accounts": len(X_val), "activated": int(y_val.sum()),
     "activation_rate": y_val.mean()},
    {"subset": "Unseen test", "accounts": len(X_test), "activated": int(y_test.sum()),
     "activation_rate": y_test.mean()},
])
print(split_table.to_string(index=False, formatters={"activation_rate": "{:.1%}".format}))
assert len(set(X_train.index) & set(X_val.index)) == 0
assert len(set(X_train.index) & set(X_test.index)) == 0
assert len(set(X_val.index) & set(X_test.index)) == 0
print("Disjointness check passed: no account appears in more than one subset.")
split_table.to_csv(RESULTS / "02_split_summary.csv", index=False)

# Fit outlier caps from training data only
WINSOR_COLS = [c for c in [
    "first_session_duration_seconds", "time_to_onboarding_completion",
    "support_resolution_time_hours", "mean_campaign_spend_zar", "kyc_days_to_verification",
    "in_app_promo_banner_impressions", "push_notifications_received", "campaign_exposures",
    "campaign_clicks", "Pre_Deposit_Engagement_Count"] if c in X_train.columns]

caps = {c: float(X_train[c].quantile(0.99)) for c in WINSOR_COLS}
def winsorise(X):
    X = X.copy()
    for c, cap in caps.items():
        X[c] = X[c].clip(upper=cap)
    return X

X_train_w, X_val_w, X_test_w = winsorise(X_train), winsorise(X_val), winsorise(X_test)
caps_table = pd.DataFrame({"column": list(caps), "train_99th_percentile_cap": list(caps.values())})
caps_table.to_csv(RESULTS / "02_winsor_caps.csv", index=False)
print("Winsorising caps (from training subset):")
caps_table.round(2).to_string(index=False)

NUM_COLS = X_train_w.select_dtypes(include="number").columns.tolist()
CAT_COLS = X_train_w.select_dtypes(exclude="number").columns.tolist()
print(f"Numeric features: {len(NUM_COLS)}   Categorical features: {len(CAT_COLS)}")

# Fit imputation, scaling, and encoding on training data
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), NUM_COLS),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="constant", fill_value="Not stated")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), CAT_COLS),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

preprocessor.fit(X_train_w)          # learns medians, means, SDs and categories from TRAINING ONLY
feature_names = preprocessor.get_feature_names_out().tolist()

def transform_to_frame(X_w):
    arr = preprocessor.transform(X_w)
    return pd.DataFrame(arr, index=X_w.index, columns=feature_names)

# Transform each split with the fitted preprocessor
X_train_p = transform_to_frame(X_train_w)
X_val_p = transform_to_frame(X_val_w)
X_test_p = transform_to_frame(X_test_w)

print(f"Model-ready feature count: {len(feature_names)}")
print("Missing values after preprocessing (must be 0):",
      int(X_train_p.isna().sum().sum() + X_val_p.isna().sum().sum() + X_test_p.isna().sum().sum()))

# Check transformed feature ranges
train_num_means = X_train_p[NUM_COLS].mean().abs().max()
print(f"Largest |mean| of scaled numeric features on TRAINING: {train_num_means:.2e}  (~0 expected)")
print("Largest |mean| of scaled numeric features on TEST (not forced to 0): "
      f"{X_test_p[NUM_COLS].mean().abs().max():.3f}")

X_train_p.to_csv(ARTIFACTS / "X_train.csv")
X_val_p.to_csv(ARTIFACTS / "X_validation.csv")
X_test_p.to_csv(ARTIFACTS / "X_test.csv")
y_train.to_frame(TARGET).to_csv(ARTIFACTS / "y_train.csv")
y_val.to_frame(TARGET).to_csv(ARTIFACTS / "y_validation.csv")
y_test.to_frame(TARGET).to_csv(ARTIFACTS / "y_test.csv")

modelling.to_csv(ARTIFACTS / "modelling_table_clean.csv")

# Save prepared splits and preprocessing artefacts
splits = pd.concat([
    pd.DataFrame({"account_id": X_train.index, "subset": "train"}),
    pd.DataFrame({"account_id": X_val.index, "subset": "validation"}),
    pd.DataFrame({"account_id": X_test.index, "subset": "test"}),
])
splits.to_csv(ARTIFACTS / "split_assignments.csv", index=False)
joblib.dump(preprocessor, ARTIFACTS / "preprocessor.joblib")
pd.Series(feature_names, name="feature").to_csv(ARTIFACTS / "feature_names.csv", index=False)

print("Saved artifacts:")
for p in sorted(ARTIFACTS.iterdir()):
    print(f"  {p.name:<28} {p.stat().st_size/1024:>8.1f} KB")
print("\nPreprocessing complete. Next: Notebook 03 (model training).")


table01_registration_onboarding      (4920, 13)
table02_initial_session_behaviour    (4906, 13)
table03_demographics_identity        (4906, 8)
table04_communication_support        (4906, 10)
table05_pre_deposit_interaction      (4906, 12)
table06_inapp_messaging_response     (4906, 9)
table08_marketing_campaigns          (8013, 10)
table09_compliance_kyc               (4906, 9)
table11_device_technical             (4906, 9)
Columns whose spelling was normalised:
table01 after exact-duplicate removal: 4,906 accounts
New missing values created by numeric coercion (should be 0): 0
PASS  registration_hour in 0-23
PASS  num_onboarding_sessions >= 1
PASS  push_notification_open_rate in [0,1]
PASS  onboarding_email_open_rate in [0,1]
PASS  promotional_email_open_rate in [0,1]
PASS  promo clicks <= impressions
PASS  support_resolution_time_hours >= 0
Campaign rows: 8,013  ->  accounts exposed: 4,906
joined table02_initial_session_behaviour    -> (4906, 20)
joined table03_demographics_identity 

# 03 - Model training

## Purpose
Train the final SS2 modelling approach on the prepared training set.

## Workflow
1. Apply SMOTE to the training set
2. Train a Random Forest
3. Train an XGBoost model
4. Train a Logistic Regression (third model)
5. Combine predictions by averaging probabilities
6. Save results for validation and reporting

## SS2 vs SS3
| Item | SS3 decision |
|---|---|
| Final model | RF + XGBoost ensemble; Logistic Regression added as the third model for the SS2 Part C statistical comparison |
| Imbalance handling | SMOTE on training data |
| Leakage control | Preprocessing fit on training only |
| Target definition | Client `activated` field |
| Split | 70/15/15 with validation set |

The validation set is used to monitor learning. The unseen test set is not touched during model training.

In [31]:
# Train the SS2 ensemble SMOTE + Random Forest + XGBoost
# Fit on train data only; validation and test stay untouched
# Save model outputs for evaluation

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

ARTIFACTS = BASE / "artifacts"
RESULTS = BASE / "results"
FIGS = RESULTS / "figures"
for _p in (ARTIFACTS, RESULTS, FIGS):
    _p.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TARGET = "activated"

# Set model parameters and training helpers
RF_PARAMS = dict(
    n_estimators=100, max_depth=15, min_samples_split=10, min_samples_leaf=5,
    max_features="sqrt", class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1,
)
XGB_PARAMS = dict(
    n_estimators=100, learning_rate=0.1, max_depth=6, min_child_weight=1,
    subsample=0.8, colsample_bytree=0.8, gamma=0, reg_lambda=1.0, reg_alpha=0,
    objective="binary:logistic", eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1,
)
SMOTE_K = 5
DECISION_THRESHOLD = 0.5

def load_split(name):
    X = pd.read_csv(ARTIFACTS / f"X_{name}.csv", index_col=0)
    y = pd.read_csv(ARTIFACTS / f"y_{name}.csv", index_col=0)[TARGET]
    return X, y

def fit_ensemble(X, y, X_monitor=None, y_monitor=None, X_val_monitor=None, y_val_monitor=None):
    """SS2 method: SMOTE on the training data, then Random Forest and XGBoost.

    Returns (rf, xgb_model, curve) where curve is the XGBoost loss history when monitors are given.
    """
# Balance the training data
    smote = SMOTE(k_neighbors=SMOTE_K, random_state=RANDOM_STATE)
    X_bal, y_bal = smote.fit_resample(X, y)

    rf = RandomForestClassifier(**RF_PARAMS).fit(X_bal, y_bal)

# Train XGBoost and record validation loss
    xgb_model = xgb.XGBClassifier(**XGB_PARAMS)
    eval_sets, names = [], []
    if X_monitor is not None and X_val_monitor is not None:
        eval_sets = [(X_monitor, y_monitor), (X_val_monitor, y_val_monitor)]
    xgb_model.fit(X_bal, y_bal, eval_set=eval_sets if eval_sets else None, verbose=False)
    curve = xgb_model.evals_result() if eval_sets else None
    return rf, xgb_model, curve

def ensemble_scores(rf, xgb_model, X):
    p_rf = rf.predict_proba(X)[:, 1]
    p_xgb = xgb_model.predict_proba(X)[:, 1]
    return p_rf, p_xgb, (p_rf + p_xgb) / 2

def metric_row(y_true, p, label, thr=DECISION_THRESHOLD):
    y_hat = (p >= thr).astype(int)
    return {
        "Model": label,
        "Accuracy": accuracy_score(y_true, y_hat),
        "Precision": precision_score(y_true, y_hat, zero_division=0),
        "Recall": recall_score(y_true, y_hat, zero_division=0),
        "F1": f1_score(y_true, y_hat, zero_division=0),
        "ROC_AUC": roc_auc_score(y_true, p),
        "PR_AUC": average_precision_score(y_true, p),
        "Brier": brier_score_loss(y_true, p),
    }

# Load training and validation splits
X_train, y_train = load_split("train")
X_val, y_val = load_split("validation")
print(f"Training subset   : {X_train.shape}  activation rate {y_train.mean():.1%}")
print(f"Validation subset : {X_val.shape}  activation rate {y_val.mean():.1%}")
print(f"Features          : {X_train.shape[1]}")

smote = SMOTE(k_neighbors=SMOTE_K, random_state=RANDOM_STATE)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
print(f"Before SMOTE : {len(y_train):,} rows, activated {y_train.sum():,} ({y_train.mean():.1%})")
print(f"After  SMOTE : {len(y_train_bal):,} rows, activated {y_train_bal.sum():,} ({y_train_bal.mean():.1%})")

# Train the Random Forest
rf_model = RandomForestClassifier(**RF_PARAMS)
rf_model.fit(X_train_bal, y_train_bal)
print("Random Forest trained with:")
for k, v in RF_PARAMS.items():
    print(f"  {k:<18} = {v}")
print(f"Trees grown: {len(rf_model.estimators_)}")

xgb_model = xgb.XGBClassifier(**XGB_PARAMS)
xgb_model.fit(
    X_train_bal, y_train_bal,
    eval_set=[(X_train, y_train), (X_val, y_val)],   # monitor on real (un-balanced) data
    verbose=False,
)

curve = xgb_model.evals_result()
loss_curve = pd.DataFrame({
    "boosting_round": np.arange(1, len(curve["validation_0"]["logloss"]) + 1),
    "train_logloss": curve["validation_0"]["logloss"],
    "validation_logloss": curve["validation_1"]["logloss"],
})
loss_curve.to_csv(RESULTS / "03_xgb_training_curve.csv", index=False)

best_round = int(loss_curve.validation_logloss.idxmin()) + 1
print(f"XGBoost trained for {len(loss_curve)} rounds")
print(f"Final train logloss      : {loss_curve.train_logloss.iloc[-1]:.4f}")
print(f"Final validation logloss : {loss_curve.validation_logloss.iloc[-1]:.4f}")
print(f"Lowest validation logloss: {loss_curve.validation_logloss.min():.4f} at round {best_round}")
print("Gap (validation minus train) at the final round: "
      f"{loss_curve.validation_logloss.iloc[-1] - loss_curve.train_logloss.iloc[-1]:.4f}")

# Train the Logistic Regression (third model, SS2 Part B method)
LR_PARAMS = dict(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)
lr_model = LogisticRegression(**LR_PARAMS)
lr_model.fit(X_train_bal, y_train_bal)
print(f"Logistic Regression trained (max_iter=1000, class_weight='balanced')")

# Generate model and ensemble probabilities
p_rf_train, p_xgb_train, p_ens_train = ensemble_scores(rf_model, xgb_model, X_train)
p_rf_val, p_xgb_val, p_ens_val = ensemble_scores(rf_model, xgb_model, X_val)
p_lr_train = lr_model.predict_proba(X_train)[:, 1]
p_lr_val = lr_model.predict_proba(X_val)[:, 1]

def scores_frame(index, actual, p_rf, p_xgb, p_lr, p_ens):
    return pd.DataFrame({
        "account_id": index,
        "actual": actual.values,
        "rf_prob": p_rf,
        "xgb_prob": p_xgb,
        "lr_prob": p_lr,
        "ensemble_prob": p_ens,
        "ensemble_label": (p_ens >= DECISION_THRESHOLD).astype(int),
    })

pred_train = scores_frame(X_train.index, y_train, p_rf_train, p_xgb_train, p_lr_train, p_ens_train)
pred_val = scores_frame(X_val.index, y_val, p_rf_val, p_xgb_val, p_lr_val, p_ens_val)
pred_train.to_csv(ARTIFACTS / "predictions_train.csv", index=False)
pred_val.to_csv(ARTIFACTS / "predictions_validation.csv", index=False)
print("Saved predictions for the training and validation subsets.")
pred_val.head()

# Evaluate training and validation predictions
rows = []
for split_name, y_s, p_rf_s, p_x_s, p_lr_s, p_e_s in [
    ("Training", y_train, p_rf_train, p_xgb_train, p_lr_train, p_ens_train),
    ("Validation", y_val, p_rf_val, p_xgb_val, p_lr_val, p_ens_val),
]:
    for label, p in [("Random Forest", p_rf_s), ("XGBoost", p_x_s),
                     ("Logistic Regression", p_lr_s), ("Ensemble", p_e_s)]:
        row = metric_row(y_s, p, label)
        row["Split"] = split_name
        rows.append(row)
training_metrics = pd.DataFrame(rows)[["Split", "Model", "Accuracy", "Precision", "Recall", "F1",
                                       "ROC_AUC", "PR_AUC", "Brier"]]
training_metrics.to_csv(RESULTS / "03_training_metrics.csv", index=False)
training_metrics.round(4)

# Calculate feature importance
feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "random_forest": rf_model.feature_importances_,
    "xgboost": xgb_model.feature_importances_,
})
feature_importance["ensemble_average"] = feature_importance[["random_forest", "xgboost"]].mean(axis=1)
feature_importance = feature_importance.sort_values("ensemble_average", ascending=False).reset_index(drop=True)
feature_importance.to_csv(RESULTS / "03_feature_importance.csv", index=False)
print("Top 15 features (ensemble average importance):")
feature_importance.head(15).round(4)

lr_coefs = pd.DataFrame({"feature": X_train.columns, "coefficient": lr_model.coef_[0]})
lr_coefs["abs_coefficient"] = lr_coefs["coefficient"].abs()
lr_coefs = lr_coefs.sort_values("abs_coefficient", ascending=False).reset_index(drop=True)
lr_coefs.to_csv(RESULTS / "03_lr_coefficients.csv", index=False)

joblib.dump(rf_model, ARTIFACTS / "rf_model.joblib")
joblib.dump(xgb_model, ARTIFACTS / "xgb_model.joblib")
joblib.dump(lr_model, ARTIFACTS / "lr_model.joblib")
print("Saved models:")
# Save fitted models and training outputs
for name in ["rf_model.joblib", "xgb_model.joblib", "lr_model.joblib"]:
    print(f"  {name:<18} {(ARTIFACTS / name).stat().st_size/1024:>8.1f} KB")
print("\nTraining complete. Next: Notebook 04 (validation on the unseen test subset).")

Training subset   : (3434, 169)  activation rate 42.7%
Validation subset : (736, 169)  activation rate 42.7%
Features          : 169
Before SMOTE : 3,434 rows, activated 1,465 (42.7%)
After  SMOTE : 3,938 rows, activated 1,969 (50.0%)
Random Forest trained with:
  n_estimators       = 100
  max_depth          = 15
  min_samples_split  = 10
  min_samples_leaf   = 5
  max_features       = sqrt
  class_weight       = balanced
  random_state       = 42
  n_jobs             = -1
Trees grown: 100
XGBoost trained for 100 rounds
Final train logloss      : 0.0755
Final validation logloss : 0.2744
Lowest validation logloss: 0.2645 at round 40
Gap (validation minus train) at the final round: 0.1989
Logistic Regression trained (max_iter=1000, class_weight='balanced')
Saved predictions for the training and validation subsets.
Top 15 features (ensemble average importance):
Saved models:
  rf_model.joblib      2804.4 KB
  xgb_model.joblib      314.7 KB
  lr_model.joblib         7.1 KB

Training compl

04 - Model validation

## Purpose
Assess the trained models on unseen data and check whether they support the business decision.

## Validation protocol
- Score the training, validation, and test subsets with the frozen models
- Compare model performance without retraining or tuning
- Keep the decision threshold at 0.5 unless validation shows a clear reason to change it
- Rank accounts by activation risk for prioritised outreach
- Test robustness with a sensitivity check on timing-sensitive features

## Metrics
Accuracy, precision, recall, F1, ROC-AUC, PR-AUC, and Brier score are reported. The main business focus is ranking and recall for the accounts most likely to stall.

In [32]:
# Evaluate the frozen model on train, validation, and test
# Check business performance, threshold choice, and ranking
# Run sensitivity checks for timing sensitive features

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

ARTIFACTS = BASE / "artifacts"
RESULTS = BASE / "results"
FIGS = RESULTS / "figures"
for _p in (ARTIFACTS, RESULTS, FIGS):
    _p.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TARGET = "activated"

# Load model settings and evaluation helpers
RF_PARAMS = dict(
    n_estimators=100, max_depth=15, min_samples_split=10, min_samples_leaf=5,
    max_features="sqrt", class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1,
)
XGB_PARAMS = dict(
    n_estimators=100, learning_rate=0.1, max_depth=6, min_child_weight=1,
    subsample=0.8, colsample_bytree=0.8, gamma=0, reg_lambda=1.0, reg_alpha=0,
    objective="binary:logistic", eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1,
)
SMOTE_K = 5
DECISION_THRESHOLD = 0.5

def load_split(name):
    X = pd.read_csv(ARTIFACTS / f"X_{name}.csv", index_col=0)
    y = pd.read_csv(ARTIFACTS / f"y_{name}.csv", index_col=0)[TARGET]
    return X, y

def fit_ensemble(X, y, X_monitor=None, y_monitor=None, X_val_monitor=None, y_val_monitor=None):
    """SS2 method: SMOTE on the training data, then Random Forest and XGBoost.

    Returns (rf, xgb_model, curve) where curve is the XGBoost loss history when monitors are given.
    """
    smote = SMOTE(k_neighbors=SMOTE_K, random_state=RANDOM_STATE)
    X_bal, y_bal = smote.fit_resample(X, y)

    rf = RandomForestClassifier(**RF_PARAMS).fit(X_bal, y_bal)

    xgb_model = xgb.XGBClassifier(**XGB_PARAMS)
    eval_sets, names = [], []
    if X_monitor is not None and X_val_monitor is not None:
        eval_sets = [(X_monitor, y_monitor), (X_val_monitor, y_val_monitor)]
    xgb_model.fit(X_bal, y_bal, eval_set=eval_sets if eval_sets else None, verbose=False)
    curve = xgb_model.evals_result() if eval_sets else None
    return rf, xgb_model, curve

def ensemble_scores(rf, xgb_model, X):
    p_rf = rf.predict_proba(X)[:, 1]
    p_xgb = xgb_model.predict_proba(X)[:, 1]
    return p_rf, p_xgb, (p_rf + p_xgb) / 2

def metric_row(y_true, p, label, thr=DECISION_THRESHOLD):
    y_hat = (p >= thr).astype(int)
    return {
        "Model": label,
        "Accuracy": accuracy_score(y_true, y_hat),
        "Precision": precision_score(y_true, y_hat, zero_division=0),
        "Recall": recall_score(y_true, y_hat, zero_division=0),
        "F1": f1_score(y_true, y_hat, zero_division=0),
        "ROC_AUC": roc_auc_score(y_true, p),
        "PR_AUC": average_precision_score(y_true, p),
        "Brier": brier_score_loss(y_true, p),
    }

# Load the saved data splits and models
X_train, y_train = load_split("train")
X_val, y_val = load_split("validation")
X_test, y_test = load_split("test")

rf_model = joblib.load(ARTIFACTS / "rf_model.joblib")
xgb_model = joblib.load(ARTIFACTS / "xgb_model.joblib")
lr_model = joblib.load(ARTIFACTS / "lr_model.joblib")

# Score the frozen models on the test set
p_rf_test, p_xgb_test, p_ens_test = ensemble_scores(rf_model, xgb_model, X_test)
p_lr_test = lr_model.predict_proba(X_test)[:, 1]
pred_test = pd.DataFrame({
    "account_id": X_test.index,
    "actual": y_test.values,
    "rf_prob": p_rf_test,
    "xgb_prob": p_xgb_test,
    "lr_prob": p_lr_test,
    "ensemble_prob": p_ens_test,
    "ensemble_label": (p_ens_test >= DECISION_THRESHOLD).astype(int),
})
pred_test.to_csv(RESULTS / "04_test_predictions.csv", index=False)
print(f"Unseen test subset scored: {len(pred_test)} accounts, activated {y_test.sum()} ({y_test.mean():.1%})")
print("Models loaded from artifacts/ — no retraining in this notebook.")

# Compare performance across all splits
rows = []
for split_name, y_s, p_rf_s, p_x_s, p_e_s, p_lr_s in [
    ("Training", y_train, *ensemble_scores(rf_model, xgb_model, X_train), lr_model.predict_proba(X_train)[:, 1]),
    ("Validation", y_val, *ensemble_scores(rf_model, xgb_model, X_val), lr_model.predict_proba(X_val)[:, 1]),
    ("Unseen test", y_test, p_rf_test, p_xgb_test, p_ens_test, p_lr_test),
]:
    for label, p in [("Random Forest", p_rf_s), ("XGBoost", p_x_s),
                     ("Logistic Regression", p_lr_s), ("Ensemble (SS2 method)", p_e_s)]:
        row = metric_row(y_s, p, label)
        row["Split"] = split_name
        rows.append(row)

metrics_all = pd.DataFrame(rows)[["Split", "Model", "Accuracy", "Precision", "Recall", "F1",
                                  "ROC_AUC", "PR_AUC", "Brier"]]
metrics_all.to_csv(RESULTS / "04_metrics_all_splits.csv", index=False)
metrics_all.round(4)

# Review test-set errors
cm = confusion_matrix(y_test, pred_test["ensemble_label"])
tn, fp, fn, tp = cm.ravel()
error_rates = pd.DataFrame([{
    "TN (correct non-activators)": tn, "FP (missed stallers)": fp,
    "FN (wasted nudges)": fn, "TP (correct activators)": tp,
    "Recall, non-activators (stall capture)": tn / (tn + fp),
    "False positive rate": fp / (fp + tn),
    "False negative rate": fn / (fn + tp),
}])
error_rates.T.rename(columns={0: "unseen test"}).to_csv(RESULTS / "04_test_error_rates.csv")
print(classification_report(y_test, pred_test["ensemble_label"],
                            target_names=["Not activated", "Activated"], digits=3))
print(error_rates.T.rename(columns={0: "unseen test"}).round(3))

# Check threshold trade-offs on validation data
thresholds = np.round(np.arange(0.30, 0.71, 0.05), 2)
val_pred = pd.read_csv(ARTIFACTS / "predictions_validation.csv")
thr_rows = []
for t in thresholds:
    lab = (val_pred["ensemble_prob"] >= t).astype(int)
    thr_rows.append({
        "threshold": t,
        "precision (activated)": precision_score(val_pred["actual"], lab, zero_division=0),
        "recall (activated)": recall_score(val_pred["actual"], lab, zero_division=0),
        "F1": f1_score(val_pred["actual"], lab, zero_division=0),
        "share flagged as stalling": 1 - lab.mean(),
    })
threshold_table = pd.DataFrame(thr_rows)
threshold_table.to_csv(RESULTS / "04_threshold_sensitivity_validation.csv", index=False)
flat = threshold_table.loc[threshold_table["threshold"].between(0.30, 0.50), "F1"]
print(f"Validation F1 for thresholds 0.30 to 0.50 ranges from {flat.min():.3f} to {flat.max():.3f}.")
print(f"Spread across this range is {flat.max() - flat.min():.3f}: a small effect. 0.50 is kept for SS2 alignment; 0.45 is marginally best and is flagged for the client.")
threshold_table.round(3)

# Measure activation ranking with decile lift
test_rank = pred_test.sort_values("ensemble_prob", ascending=False).reset_index(drop=True)
test_rank["decile"] = pd.qcut(test_rank.index, 10, labels=range(1, 11))   # 1 = most likely to activate
base_rate = test_rank["actual"].mean()

decile = (test_rank.groupby("decile", observed=True)
          .agg(accounts=("actual", "size"),
               activated=("actual", "sum"),
               mean_predicted_prob=("ensemble_prob", "mean"))
          .reset_index())
decile["observed_activation_rate"] = decile["activated"] / decile["accounts"]
decile["lift_vs_average"] = decile["observed_activation_rate"] / base_rate
decile.to_csv(RESULTS / "04_decile_lift_test.csv", index=False)
print(f"Overall activation rate (test): {base_rate:.1%}")
decile.round(3)

# Measure coverage of non-activating accounts
never_activated = (pred_test["actual"] == 0)
cut = pred_test["ensemble_prob"].quantile(0.30)
flagged = pred_test["ensemble_prob"] <= cut

coverage = pd.DataFrame([{
    "accounts flagged for outreach (lowest 30% score)": int(flagged.sum()),
    "never-activated accounts in test": int(never_activated.sum()),
    "never-activated accounts reached by outreach": int((flagged & never_activated).sum()),
    "coverage of stalling accounts": (flagged & never_activated).sum() / never_activated.sum(),
    "precision of the outreach list (share truly stalling)": (flagged & never_activated).sum() / flagged.sum(),
    "random-targeting precision (base rate)": never_activated.mean(),
}])
coverage.T.rename(columns={0: "unseen test"}).to_csv(RESULTS / "04_outreach_coverage_test.csv")
print("Outreach list built from the model vs targeting at random:")
coverage.T.rename(columns={0: "unseen test"}).round(3)

# Define timing-sensitive feature scenarios
def drop_features(X, names=(), prefixes=()):
    keep = [c for c in X.columns if c not in names and not any(c.startswith(p) for p in prefixes)]
    return X[keep]

S2_DROP_NAMES = ["first_7day_session_count", "first_30day_session_count", "Early_Session_Share"]
S2_DROP_PREFIX = ["kyc_"]
S3_EXTRA_NAMES = ["search_page_visits", "watchlist_created", "watchlist_items", "detailed_stock_views",
                  "price_alert_set", "comparison_tool_used", "trading_simulation_engaged",
                  "deposit_page_views", "Pre_Deposit_Engagement_Count"]

# Run sensitivity analysis
scenarios = {
    "S1 full model": (X_train, X_test),
    "S2 no timing-sensitive snapshots": (drop_features(X_train, S2_DROP_NAMES, S2_DROP_PREFIX),
                                         drop_features(X_test, S2_DROP_NAMES, S2_DROP_PREFIX)),
    "S3 also no pre-deposit counts": (drop_features(X_train, S2_DROP_NAMES + S3_EXTRA_NAMES, S2_DROP_PREFIX),
                                      drop_features(X_test, S2_DROP_NAMES + S3_EXTRA_NAMES, S2_DROP_PREFIX)),
}

sens_rows = []
for label, (Xtr, Xte) in scenarios.items():
    rf_s, xgb_s, _ = fit_ensemble(Xtr, y_train)
    _, _, p_e = ensemble_scores(rf_s, xgb_s, Xte)
    row = metric_row(y_test, p_e, label)
    row["features"] = Xtr.shape[1]
    sens_rows.append(row)
sensitivity = pd.DataFrame(sens_rows)[["Model", "features", "Accuracy", "Precision", "Recall", "F1",
                                       "ROC_AUC", "PR_AUC", "Brier"]].rename(columns={"Model": "scenario"})
sensitivity.to_csv(RESULTS / "04_sensitivity_analysis.csv", index=False)
sensitivity.round(4)

# Verify evaluation and leakage safeguards
checks = {
    "Subsets are disjoint (no account in two subsets)":
        len(set(X_train.index) & set(X_test.index)) == 0
        and len(set(X_val.index) & set(X_test.index)) == 0
        and len(set(X_train.index) & set(X_val.index)) == 0,
    "Unseen test subset contains only accounts never used in fitting":
        set(X_test.index).isdisjoint(set(pd.read_csv(ARTIFACTS / "X_train.csv", index_col=0).index)),
    "Test predictions cover every test account exactly once":
        pred_test["account_id"].is_unique and len(pred_test) == len(X_test),
    "No target column is present among the predictors":
        TARGET not in X_test.columns,
    "Preprocessor was fitted on the training subset only (scaler saw training rows)":
        int(joblib.load(ARTIFACTS / "preprocessor.joblib").named_transformers_["num"].named_steps["scale"].n_samples_seen_) == len(X_train),
    "Test score is not better than training score by more than 0.10 ROC-AUC (overfit guard)":
        (roc_auc_score(y_train, ensemble_scores(rf_model, xgb_model, X_train)[2]) - roc_auc_score(y_test, p_ens_test)) < 0.10,
}
for label, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {label}")
    assert ok, f"integrity check failed: {label}"
print("\nAll integrity checks passed.")


Unseen test subset scored: 736 accounts, activated 314 (42.7%)
Models loaded from artifacts/ — no retraining in this notebook.
               precision    recall  f1-score   support

Not activated      0.926     0.825     0.872       422
    Activated      0.794     0.911     0.849       314

     accuracy                          0.861       736
    macro avg      0.860     0.868     0.860       736
 weighted avg      0.870     0.861     0.862       736

                                        unseen test
TN (correct non-activators)                 348.000
FP (missed stallers)                         74.000
FN (wasted nudges)                           28.000
TP (correct activators)                     286.000
Recall, non-activators (stall capture)        0.825
False positive rate                           0.175
False negative rate                           0.089
Validation F1 for thresholds 0.30 to 0.50 ranges from 0.871 to 0.879.
Spread across this range is 0.008: a small effect. 0.5